# Manual CCZ Optimization

This notebook uses the same manual-block API for a three-qubit CCZ-style phase gate. The blocks are grouped by the number of atoms initially in `|1>` and weighted by the binomial multiplicities `1, 3, 3, 1`.

The Hamiltonian definitions are intentionally written in the notebook. This keeps the core package independent of a specific symmetry model.

In [ ]:
from __future__ import annotations

from math import comb
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

project_root = Path.cwd()
if project_root.name == "examples":
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from time_optimal_grape import (
    AveragePhaseGateFidelity,
    ControlLayout,
    ControlValues,
    GrapeOptimizer,
    GrapeProblem,
    ManualHamiltonianBlock,
    OptimizerSettings,
    TimeContinuationScanner,
    TimeScanSettings,
    assemble_parameters,
    basis_state,
    phase_sample_axis,
    random_control_profile,
    trace_populations,
    unwrap_phase_profile,
    zero_control_profile,
)

## Manual Collective Hamiltonian and Fidelity

For a block with $n$ participating atoms, the basis is indexed by the number of Rydberg excitations,

$$
|0_r\rangle, |1_r\rangle, \ldots, |n_r\rangle.
$$

The collective coupling between adjacent ladder states is

$$
H_{m,m+1}(\phi_k) = \frac{\Omega e^{i\phi_k}}{2}\sqrt{(n-m)(m+1)},\quad m=0,\ldots,n-1.
$$

The interaction energy is supplied manually here as

$$
E_m = \begin{cases}
0, & m < 2,\\
(m-1)B, & m \ge 2.
\end{cases}
$$

The CCZ phase rule in this simple single-species example is

$$
\xi_n = n\theta + \pi\,\delta_{n,3}.
$$

The block weights are $\binom{3}{n}$, and the averaged phase-gate fidelity uses Hilbert dimension $d=8$.

In [ ]:
def interaction_energy(rydberg_count: int, blockade: float) -> float:
    if rydberg_count < 2:
        return 0.0
    return (rydberg_count - 1) * blockade


def collective_coupling(atom_count: int, rydberg_count: int, omega: float, phi: float) -> complex:
    multiplicity = np.sqrt((atom_count - rydberg_count) * (rydberg_count + 1))
    return multiplicity * omega * np.exp(1j * phi) / 2.0


def collective_coupling_derivative(atom_count: int, rydberg_count: int, omega: float, phi: float) -> complex:
    return 1j * collective_coupling(atom_count=atom_count, rydberg_count=rydberg_count, omega=omega, phi=phi)


def make_collective_hamiltonian(atom_count: int, omega: float, blockade: float):
    dimension = atom_count + 1

    def hamiltonian(control_values: ControlValues, sample_index: int) -> np.ndarray:
        phi = control_values.time_controls["phi"][sample_index]
        matrix = np.zeros((dimension, dimension), dtype=np.complex128)
        for rydberg_count in range(dimension):
            matrix[rydberg_count, rydberg_count] = interaction_energy(rydberg_count, blockade)
        for rydberg_count in range(atom_count):
            drive = collective_coupling(atom_count, rydberg_count, omega, phi)
            matrix[rydberg_count, rydberg_count + 1] = drive
            matrix[rydberg_count + 1, rydberg_count] = np.conj(drive)
        return matrix

    return hamiltonian


def make_collective_derivative(atom_count: int, omega: float):
    dimension = atom_count + 1

    def derivative(control_values: ControlValues, sample_index: int) -> np.ndarray:
        phi = control_values.time_controls["phi"][sample_index]
        matrix = np.zeros((dimension, dimension), dtype=np.complex128)
        for rydberg_count in range(atom_count):
            drive_derivative = collective_coupling_derivative(atom_count, rydberg_count, omega, phi)
            matrix[rydberg_count, rydberg_count + 1] = drive_derivative
            matrix[rydberg_count + 1, rydberg_count] = np.conj(drive_derivative)
        return matrix

    return derivative


def make_trivial_hamiltonian(control_values: ControlValues, sample_index: int) -> np.ndarray:
    return np.zeros((1, 1), dtype=np.complex128)


def make_trivial_derivative(control_values: ControlValues, sample_index: int) -> np.ndarray:
    return np.zeros((1, 1), dtype=np.complex128)


def make_ccz_blocks(omega: float, blockade: float) -> tuple[ManualHamiltonianBlock, ...]:
    blocks: list[ManualHamiltonianBlock] = []
    for atom_count in range(4):
        dimension = atom_count + 1
        hamiltonian = make_trivial_hamiltonian if atom_count == 0 else make_collective_hamiltonian(atom_count, omega, blockade)
        derivative = make_trivial_derivative if atom_count == 0 else make_collective_derivative(atom_count, omega)
        phase_offset = np.pi if atom_count == 3 else 0.0
        blocks.append(
            ManualHamiltonianBlock(
                name=f"n1_{atom_count}",
                weight=comb(3, atom_count),
                initial_state=basis_state(dimension, 0),
                target_state=basis_state(dimension, 0),
                hamiltonian=hamiltonian,
                derivatives={"phi": derivative},
                phase_offset=phase_offset,
                local_phase_coefficients={"theta": float(atom_count)},
            )
        )
    return tuple(blocks)

## Evaluate a Pulse and Optimize One Fixed Duration

All durations in this project are dimensionless. We plot them as $T\Omega$ or $t\Omega$.

In [ ]:
samples = 60
duration = 14.0
omega = 1.0
blockade = 100.0

layout = ControlLayout(time_control_names=("phi",), local_phase_names=("theta",), samples=samples)
settings = OptimizerSettings(method="BFGS", max_iterations=400, gradient_tolerance=1e-8, display=False)
problem = GrapeProblem(
    blocks=make_ccz_blocks(omega=omega, blockade=blockade),
    control_layout=layout,
    duration=duration,
    objective=AveragePhaseGateFidelity(hilbert_dimension=8),
    optimizer_settings=settings,
)

rng = np.random.default_rng(11)
initial_parameters = assemble_parameters(
    layout=layout,
    time_controls={"phi": random_control_profile(samples=samples, low=-0.05, high=0.05, rng=rng)},
    local_phases={"theta": 0.0},
)
zero_parameters = assemble_parameters(
    layout=layout,
    time_controls={"phi": zero_control_profile(samples=samples)},
    local_phases={"theta": 0.0},
)

initial_infidelity, _initial_gradient = GrapeOptimizer(problem).evaluate(initial_parameters)
zero_infidelity, _zero_gradient = GrapeOptimizer(problem).evaluate(zero_parameters)
print(f"random initial infidelity: {initial_infidelity:.6e}")
print(f"zero-phase infidelity: {zero_infidelity:.6e}")

In [ ]:
result = GrapeOptimizer(problem).optimize(initial_parameters)
print(f"success: {result.success}")
print(f"message: {result.message}")
print(f"nominal infidelity: {1.0 - result.fidelity:.6e}")

In [ ]:
optimized_controls = layout.unpack(result.parameters)
time_axis = phase_sample_axis(duration=duration, samples=samples)
plt.plot(time_axis, unwrap_phase_profile(optimized_controls.time_controls["phi"]))
plt.xlabel(r"dimensionless time $t\Omega$")
plt.ylabel("unwrapped phase")
plt.title("Optimized CCZ phase profile")
plt.grid(True)
plt.show()

## Trace Populations

Here we inspect the block with three atoms initially in `|1>`, using the collective Rydberg-excitation basis.

In [ ]:
ccz_controls = layout.unpack(result.parameters)
block_n3 = next(block for block in problem.blocks if block.name == "n1_3")
population_trace = trace_populations(
    block=block_n3,
    control_values=ccz_controls,
    duration=duration,
    samples=samples,
    states={
        "0 Rydberg": basis_state(4, 0),
        "1 Rydberg": basis_state(4, 1),
        "2 Rydberg": basis_state(4, 2),
        "3 Rydberg": basis_state(4, 3),
    },
)

for label, values in population_trace.populations.items():
    plt.plot(population_trace.times, values, label=label)
plt.xlabel(r"dimensionless time $t\Omega$")
plt.ylabel("population")
plt.title("CCZ n1=3 block populations")
plt.legend()
plt.grid(True)
plt.show()

## Time-Continuation Scan

The scan starts from a longer duration and uses each optimized pulse as the initial guess for the next shorter duration. The plot is sorted so that $T\Omega$ increases from left to right.

In [ ]:
scan_settings = TimeScanSettings(
    start_duration=16.0,
    stop_duration=10.0,
    duration_step=1.0,
    infidelity_threshold=1e-3,
    stop_after_threshold_failure=True,
)
scan_problem = problem.with_duration(scan_settings.start_duration)
scan_points = TimeContinuationScanner(scan_problem, scan_settings).scan(initial_parameters)

durations = np.array([point.duration for point in scan_points], dtype=np.float64)
infidelities = np.array([point.result.infidelity for point in scan_points], dtype=np.float64)
for duration_value, infidelity_value in zip(durations, infidelities, strict=True):
    print(f"T*Omega={duration_value:.3f}, infidelity={infidelity_value:.6e}")

order = np.argsort(durations)
plt.semilogy(durations[order], infidelities[order], marker="o")
plt.xlabel(r"dimensionless pulse duration $T\Omega$")
plt.ylabel("infidelity")
plt.title("CCZ time-continuation scan")
plt.grid(True)
plt.show()